# Investment Research Agent (IR-Agent)

## Step 1: 에이전트 설계

### 이름
**IR-Agent** (Investment Research Agent)

### 목적
10-Q / 사업보고서 PDF를 입력받아, LangGraph 파이프라인을 통해 자동으로 분석하고 투자자 관점의 구조화된 요약 보고서를 생성한다.

### 핵심 기능 (3가지)
1. **PDF 파싱** — pypdf로 보고서 전문 텍스트를 추출한다.
2. **재무 분석** — LLM이 매출·영업이익·현금흐름·부채비율·핵심 리스크 등 투자 핵심 지표를 구조화하여 추출한다.
3. **투자 인사이트 리포트 생성** — 기회 요인·리스크 요인·투자 의견을 포함한 최종 보고서를 마크다운 형식으로 출력한다.

### 그래프 구조

```
START
  │
  ▼
┌─────────────────┐
│  load_document  │  PDF 경로 입력 → 텍스트 추출 (pypdf)
└────────┬────────┘
         │ raw_text
         ▼
┌──────────────────────┐
│  analyze_financials  │  LLM → 재무 지표 추출 (매출/영업이익/현금흐름/리스크)
└────────┬─────────────┘
         │ financial_data
         ▼
┌─────────────────┐
│ generate_report │  LLM → 투자 인사이트 최종 보고서 생성
└────────┬────────┘
         │ report
         ▼
        END
```

### State 정의
```python
class ResearchState(TypedDict):
    pdf_path: str        # 입력: PDF 파일 경로
    raw_text: str        # Node 1 출력: 추출된 전체 텍스트
    financial_data: str  # Node 2 출력: 재무 지표 분석 결과
    report: str          # Node 3 출력: 최종 투자 보고서
```

## Step 2: 기초 구축

In [ ]:
# 패키지 임포트
import os
from pathlib import Path
from typing import TypedDict

from dotenv import load_dotenv
from pypdf import PdfReader
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

load_dotenv()
print("패키지 로드 완료")

In [ ]:
# ── State 정의 ──────────────────────────────────────────────────────────────
class ResearchState(TypedDict):
    pdf_path: str        # 입력: PDF 파일 경로
    raw_text: str        # Node 1 출력: 추출된 전체 텍스트
    financial_data: str  # Node 2 출력: 재무 지표 분석 결과
    report: str          # Node 3 출력: 최종 투자 보고서

print("State 정의 완료:", ResearchState.__annotations__)

In [ ]:
# ── LLM 초기화 ──────────────────────────────────────────────────────────────
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    max_tokens=4096,
)

# PDF 1회에 처리할 최대 글자 수 (토큰 초과 방지)
MAX_CHARS = 60_000

print("LLM 초기화 완료:", llm.model_name)

In [ ]:
# ── Node 1: load_document ────────────────────────────────────────────────────
def load_document(state: ResearchState) -> ResearchState:
    """PDF 파일을 읽어 텍스트를 추출한다."""
    print(f"[Node 1] PDF 로딩: {state['pdf_path']}")

    reader = PdfReader(state["pdf_path"])
    pages = [page.extract_text() or "" for page in reader.pages]
    full_text = "\n".join(pages)

    # 토큰 초과 방지: 앞부분 MAX_CHARS 글자만 사용
    if len(full_text) > MAX_CHARS:
        full_text = full_text[:MAX_CHARS]
        print(f"  → 텍스트 길이 초과, 앞 {MAX_CHARS:,}자로 잘라냄")

    print(f"  → 총 {len(reader.pages)}페이지, 추출 글자 수: {len(full_text):,}")
    return {**state, "raw_text": full_text}


# ── Node 2: analyze_financials ───────────────────────────────────────────────
def analyze_financials(state: ResearchState) -> ResearchState:
    """LLM을 사용해 재무 핵심 지표를 추출한다."""
    print("[Node 2] 재무 지표 분석 중...")

    messages = [
        SystemMessage(content=(
            "당신은 전문 투자 리서치 애널리스트입니다. "
            "주어진 사업보고서(10-Q) 텍스트에서 투자 판단에 필요한 핵심 재무 정보를 추출하세요.\n\n"
            "반드시 아래 항목을 마크다운 형식으로 구조화하여 답변하세요:\n"
            "## 회사 기본 정보\n"
            "## 핵심 재무 지표 (매출, 영업이익, 순이익, EBITDA, 현금흐름)\n"
            "## 전년 대비 주요 변화\n"
            "## 재무 건전성 (부채비율, 유동비율, 현금 보유량)\n"
            "## 경영진이 언급한 주요 리스크 및 불확실성"
        )),
        HumanMessage(content=f"다음은 사업보고서 내용입니다:\n\n{state['raw_text']}")
    ]

    response = llm.invoke(messages)
    financial_data = response.content

    print(f"  → 재무 분석 완료 (출력 길이: {len(financial_data):,}자)")
    return {**state, "financial_data": financial_data}


# ── Node 3: generate_report ──────────────────────────────────────────────────
def generate_report(state: ResearchState) -> ResearchState:
    """재무 분석 결과를 바탕으로 투자 인사이트 보고서를 생성한다."""
    print("[Node 3] 투자 보고서 생성 중...")

    messages = [
        SystemMessage(content=(
            "당신은 기관 투자자를 위한 시니어 주식 애널리스트입니다. "
            "주어진 재무 분석 데이터를 바탕으로 투자자가 즉시 활용할 수 있는 "
            "인사이트 보고서를 한국어로 작성하세요.\n\n"
            "보고서는 반드시 아래 구조를 따르세요:\n"
            "# [회사명] 투자 리서치 요약 보고서\n\n"
            "## 1. 회사 개요\n"
            "## 2. 핵심 재무 지표 요약 (표 형식 권장)\n"
            "## 3. 성장 기회 요인 (Bullish Case)\n"
            "## 4. 리스크 요인 (Bearish Case)\n"
            "## 5. 종합 투자 의견\n"
            "## 6. 주목해야 할 다음 촉매 (Next Catalyst)\n"
            "---\n"
            "*본 보고서는 AI가 생성한 참고 자료이며 투자 권유가 아닙니다.*"
        )),
        HumanMessage(content=f"다음 재무 분석 결과를 기반으로 보고서를 작성하세요:\n\n{state['financial_data']}")
    ]

    response = llm.invoke(messages)
    report = response.content

    print(f"  → 보고서 생성 완료 (출력 길이: {len(report):,}자)")
    return {**state, "report": report}


print("노드 함수 정의 완료: load_document / analyze_financials / generate_report")

In [ ]:
# ── 그래프 빌드 ──────────────────────────────────────────────────────────────
builder = StateGraph(ResearchState)

# 노드 등록
builder.add_node("load_document", load_document)
builder.add_node("analyze_financials", analyze_financials)
builder.add_node("generate_report", generate_report)

# 엣지 연결
builder.add_edge(START, "load_document")
builder.add_edge("load_document", "analyze_financials")
builder.add_edge("analyze_financials", "generate_report")
builder.add_edge("generate_report", END)

# 컴파일
graph = builder.compile()

print("그래프 컴파일 완료")
print("노드:", ["load_document", "analyze_financials", "generate_report"])
print("엣지: START → load_document → analyze_financials → generate_report → END")

In [ ]:
# ── 그래프 시각화 (선택) ──────────────────────────────────────────────────────
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print("그래프 시각화 생략 (graphviz 미설치 시 정상)")

In [ ]:
# ── 실행: data/써클.pdf 분석 ──────────────────────────────────────────────────
PDF_PATH = str(Path("data/써클.pdf").resolve())

initial_state: ResearchState = {
    "pdf_path": PDF_PATH,
    "raw_text": "",
    "financial_data": "",
    "report": "",
}

print("=" * 60)
print("IR-Agent 실행 시작")
print("=" * 60)

result = graph.invoke(initial_state)

print("\n" + "=" * 60)
print("모든 노드 실행 완료")
print("=" * 60)

In [ ]:
# ── 최종 보고서 출력 ─────────────────────────────────────────────────────────
from IPython.display import Markdown, display

display(Markdown(result["report"]))

In [ ]:
# ── 중간 결과 확인 (재무 지표 분석) ──────────────────────────────────────────
print("[Node 2 출력 - 재무 지표 분석]")
print("-" * 60)
display(Markdown(result["financial_data"]))